# Lab 10 · V-SKNN

The runnable companion of the lab page, `docs/labs/10-vsknn.md` (also on the docs site). The page explains each step,
asks the questions and holds the hints and solutions; this notebook has the code.

- Run the setup cell first, every time the kernel starts (after a crash: **Restart**, then setup again).
- Cells that fit models take seconds to minutes on MovieLens. Run them where you train: on the rented box (VS Code's
  Remote - SSH extension opens this notebook there), or on the laptop if you allow training there.
- Cells that only read stored results (the baseline, `lab.compare`, `lab.segments`) are fine anywhere, once the box's
  results are fetched (`./scripts/fetch_lab_results.sh`).
- Save the notebook **without outputs** before you commit (the **...** menu → Clear All Outputs).

In [ ]:
from recbench import lab

lab.setup()
DATASET = "movielens-25m"
METHOD = "vsknn"

## Your starting point

The lab baseline on every dataset (stored results; nothing is trained here).

In [ ]:
lab.baseline(METHOD)

## Level 1: reproduce and read

### 1.1 Reproduce the baseline's best setting

Fit the baseline's best setting on the validation fold, in this notebook. The NDCG@10 must equal the baseline's validation score.

In [ ]:
best = lab.baseline(METHOD, DATASET)["best_settings"]
data, split = lab.load(DATASET)
model = lab.fit(METHOD, data, **best)
result = lab.evaluate(model, data, split)
print(result)
print("the baseline's best trial:", lab.baseline(METHOD, DATASET)["validation"])

### 1.2 Read the code

The questions are on the lab page; answer them with the code open next to this notebook.

### 1.3 Look inside

In [ ]:
import numpy as np

print(model.fit_info)  # how many past sessions there are
user = int(np.flatnonzero(data.user_lengths > 5)[0])
items, weights = model._profile(user)  # the user's last session, latest first, with position weights
for item, w in zip(items, weights):
    print(round(float(w), 2), data.item_text[item] or data.item_ids[item])

**Exercise.** How long are the past sessions? Compute the mean and the largest number of items per session.

<details><summary>Hint</summary>

`model.sessions` is a session x item matrix: row lengths are `np.diff(model.sessions.indptr)`.

</details>

<details><summary>Solution</summary>

```python
per_session = np.diff(model.sessions.indptr)
print(per_session.mean(), per_session.max())
```
On MovieLens a "session" is a sitting in which someone rated films, so sessions can be long.

</details>

In [ ]:
# Your code here

## Level 2: understand each setting

For each sweep: **write your prediction in the cell below it first**, then run the sweep. Every other setting stays at the baseline's best. The lab page explains what to look for.

In [ ]:
table = lab.sweep(METHOD, DATASET, "vsknn_last_n", [None, 10, 50], start="best")
lab.plot_sweep(table)
table

**Your prediction for `vsknn_last_n`, and what you saw:** ...

In [ ]:
table = lab.sweep(METHOD, DATASET, "vsknn_weighting", ['div', 'linear', 'same'], start="best")
lab.plot_sweep(table)
table

**Your prediction for `vsknn_weighting`, and what you saw:** ...

## Level 3: data tricks

Experiments run on the rented box over all five datasets, for example:

```bash
./scripts/run_lab_box.sh vsknn:no-window
```

Then fetch the results (`./scripts/fetch_lab_results.sh <host>`) and compare here. `no-window` is ready to run; `session-gap`
needs a small code change first (the lab page has the hint and the folded solution). After writing it, run its test:
`pytest -q tests/test_vsknn_variant.py`.

In [ ]:
lab.compare(METHOD, f"{METHOD}:no-window")

In [ ]:
lab.compare(METHOD, f"{METHOD}:session-gap")

## Level 4: one change from the literature

Write the variant (the lab page explains the idea, with hints and the folded solution), test it, uncomment
`stan-recency` in `labs/10-vsknn/experiments.yaml`, run it on the box, fetch, then compare and look at the segments.

In [ ]:
lab.compare(METHOD, f"{METHOD}:stan-recency")

In [ ]:
parts = lab.segments(METHOD, f"{METHOD}:stan-recency", DATASET)
lab.plot_segments(parts, "activity")

In [ ]:
parts["items"]

## Record your results

Fill in the table at the end of the lab page, then rebuild the scoreboard: `python -m recbench.lab scoreboard --docs`.